In [2]:
import boto3


In [3]:
client = boto3.client("sagemaker", region_name="ap-south-1")



In [4]:
resp = client.list_pipelines(MaxResults=100)
print("Pipelines found:", len(resp.get("PipelineSummaries", [])))
for p in resp.get("PipelineSummaries", []):
    print(p["PipelineName"], p["PipelineArn"], p["CreatedTime"])

Pipelines found: 1


KeyError: 'CreatedTime'

In [4]:
import os
import boto3
import sagemaker
from pipeline import get_pipeline   # adjust import path to where your file is

region = "ap-south-1"
default_bucket = "beatit-ai-data"   # or leave None to let session pick default
pipeline_name = "beatit-ai-churn-pipeline"  # choose desired name

# get role
sess = sagemaker.session.Session(boto_session=boto3.Session(region_name=region))
role = sagemaker.session.get_execution_role(sess)  # works if running in Studio/notebook with role
# OR explicitly set role_arn = "arn:aws:iam::<account>:role/YourSageMakerRole"

pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=default_bucket,
    pipeline_name=pipeline_name,
    base_job_prefix="beatit-ai-churn",
    processing_instance_type="ml.m5.xlarge",
    training_instance_type="ml.m5.xlarge",
)

# Register/create the pipeline in SageMaker (this creates the Pipeline resource)
pipeline.upsert(role_arn=role)
print("Pipeline upserted:", pipeline.name)


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading analysis config to {s3_uri}.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading 

Pipeline upserted: beatit-ai-churn-pipeline


In [7]:
# run this in the same environment where you can import your pipeline code
from pipeline import get_pipeline
import boto3, sagemaker

region = "ap-south-1"
role = sagemaker.session.get_execution_role()   # or pass ARN if running locally
pipeline = get_pipeline(region=region, role=role, default_bucket="beatit-ai-data", pipeline_name="beatit-ai-churn-pipeline")

# list parameters and defaults
empty_params = []
for p in pipeline.parameters:
    # ParameterString has attribute default_value; other types differ
    default = getattr(p, "default_value", None)
    #print(f"Name: {p.name}, Type: {type(p).__name__}, Default: {repr(default)}")
    if default == "":
        empty_params.append(p.name)

print("Parameters with empty-string defaults:", empty_params)


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading analysis config to {s3_uri}.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading 

Parameters with empty-string defaults: ['DataQualitySuppliedStatistics', 'DataQualitySuppliedConstraints', 'DataBiasSuppliedBaselineConstraints', 'ModelQualitySuppliedStatistics', 'ModelQualitySuppliedConstraints', 'ModelBiasSuppliedBaselineConstraints', 'ModelExplainabilitySuppliedBaselineConstraints']


In [8]:
import sagemaker
session = sagemaker.Session()
session.default_bucket()


INFO:sagemaker:Created S3 bucket: sagemaker-ap-south-1-751081874324


'sagemaker-ap-south-1-751081874324'

In [1]:
from pipeline import get_pipeline
import sagemaker, boto3

region = "ap-south-1"
sess = sagemaker.session.Session(boto_session=boto3.Session(region_name=region))
role = sagemaker.session.get_execution_role(sess)  # or set explicit ARN

pipeline = get_pipeline(region=region, role=role, default_bucket=None, pipeline_name="beatit-ai-churn-pipeline", base_job_prefix="BeatItAI-Churn")
pipeline.upsert(role_arn=role)
execution = pipeline.start(parameters={"InputDataUrl": "s3://beatit-ai-data/raw/", "ModelApprovalStatus": "PendingManualApproval"})
print("Started execution:", execution.arn)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading analysis config to {s3_uri}.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python

Started execution: arn:aws:sagemaker:ap-south-1:751081874324:pipeline/beatit-ai-churn-pipeline/execution/j0dt90ml3xje
